<a href="https://colab.research.google.com/github/santed7/Data-Science-Cohort-20/blob/main/Capstone%20OrgTrustQt%20Project/Organizational_Trust_Quotient_First_Iteration_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# -----------------------------------------------------------------------------
# Author: Dr. Vernon T. Cox
#
# This code was created by Dr. Vernon Cox using the trademarked
# Organizational Trust Quotient® (OTQ) framework and documentation.
#
# Coding structure, implementation support, and explanatory assistance
# were provided with the aid of OpenAI tools.
#
# The Organizational Trust Quotient (OTQ) is a proprietary and trademarked
# methodology. All underlying concepts, scoring logic, and interpretive
# frameworks remain the intellectual property of Dr. Vernon Cox.
# -----------------------------------------------------------------------------

# OTQ Communication Stream Regression (Layman-Commented)

This notebook is a **data blender + scorekeeper + predictor** for Organizational Trust Quotient (OTQ)-style communication data.

## What it does (plain English)
1. Looks in an input folder for exported communication logs (**Slack / Teams / Outlook Calendar**).
2. Converts them into **one standardized event table** (one row per communication event).
3. Assigns each event an **Engagement Score (EU)** using a customizable scoring rule.
4. Summarizes events by **day** (daily totals and averages).
5. Builds a simple **proxy outcome** (median “response time”) based on time gaps between messages.
6. Trains a regression model to see if engagement signals can predict the outcome.
7. Saves plots + a daily CSV to an output folder.

---

## Quick start
1. Put your export files into a folder, e.g. `./data/`
2. Run the **Configuration** cell and set:
   - `INPUT_DIR = "./data"`
   - `OUT_DIR = "./out"`
3. Run all cells top to bottom.

> Tip: File names matter. If your filenames include `slack`, `teams`, or `outlook` / `calendar`, the notebook routes them automatically.


## Input file checklist (so the notebook can read your exports)

### Slack (CSV or JSON/JSONL)
**Minimum needed:**
- timestamp column: `ts` or `timestamp` (Slack often uses epoch seconds like `1700000000.1234`)
- message text: `text`
- sender/user: `user` (optional but helpful)
- channel/room: `channel` (optional but helpful)

### Teams (CSV)
Teams exports vary widely. The parser tries to “guess” columns.
**If you can, include:**
- `timestamp` or `date` or `start`
- `subject` or `title`
- `organizer` or `from`
- `duration` (minutes) (optional)
- `participants` / `attendees` / `participant_count` (optional)

### Outlook Calendar (CSV export)
**Best columns (common in Outlook exports):**
- `Subject`
- `Start` and `End`
- `Organizer`
- `Required Attendees`
- `Optional Attendees`

---

## Common errors (and what they mean)

- **“Input folder not found”**  
  The path in `INPUT_DIR` is wrong or the folder doesn't exist.

- **“No CSV/JSON found”**  
  The folder exists but contains no `.csv`, `.json`, or `.jsonl` files.

- **“No parsable files found”**  
  The notebook found files, but none matched the naming rules or supported formats.  
  Fix: rename files to include `slack`, `teams`, or `outlook` / `calendar`.

- **“Not enough merged daily rows to run regression”**  
  Your daily features and outcome don’t overlap enough days (need ~10+ days).  
  Fix: add more history, or replace the proxy outcome with a real KPI series.


In [ ]:
# =========================
# Configuration (edit me)
# =========================

# Folder containing your exports/logs (csv/json/jsonl)
INPUT_DIR = "./data"

# Output folder for plots + tables
OUT_DIR = "./out"

# Optional: set to True if you want extra debug printing
DEBUG = False


In [ ]:
"""
Layman-commented OTQ communication stream regression notebook.

This cell defines all functions and the pipeline runner [series of steps that makes them happen in sequence].
"""

from __future__ import annotations

"""
otq_comm_stream_regression.py

BIG PICTURE (Layman version):
This script is a “data blender + scorekeeper + predictor”.

1) It reads communication exports (Slack / Teams / Outlook calendar)
2) It standardizes them into one common table of events
3) It calculates an OTQ-style engagement score (Event Units / EU)
4) It summarizes the data by day (daily totals and averages)
5) It creates (or expects) an “outcome” to predict
   - Here: a proxy outcome = estimated response time in minutes
6) It trains a regression model to predict that outcome
7) It saves plots + a CSV to an output folder

Usage example:
  python otq_comm_stream_regression.py --inputs ./data --out ./out

NOTE:
- This is a starter framework. The parsing rules and OTQ scoring weights are designed
  to be customized to match your exact OTQ policy and your file formats.
"""
# -----------------------------
# Imports (tools the notebook needs)
# -----------------------------
import argparse   # lets you run the script with command-line options like --inputs
import json       # used to read JSON/JSONL files (common for Slack exports)
import math       # used for safe math like log calculations
import re         # used for pattern matching (timestamps, splitting emails, etc.)
from datetime import timezone
from pathlib import Path  # safe file/folder handling across operating systems
from typing import Any, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# Machine learning tools
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split


# ============================================================
# 1) Small “safety” helper functions
#    These protect the script from crashing when data is messy
# ============================================================

def safe_int(x, default=0) -> int:
    """
    Converts something to an integer safely.
    If it fails (blank, weird text, etc.), returns default instead.
    """
    try:
        return int(x)
    except Exception:
        return default


def safe_float(x, default=0.0) -> float:
    """
    Converts something to a float safely.
    If it fails, returns default instead.
    """
    try:
        return float(x)
    except Exception:
        return default


def parse_datetime(value: Any) -> Optional[pd.Timestamp]:
    """
    Tries to turn many different “timestamp-looking” values into
    a real datetime object (in UTC timezone).

    Why this matters:
    - Slack, Teams, and Outlook export dates in different formats.
    - We need them all in one common format to sort and group by day.
    """
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    # If it’s already a pandas Timestamp, use it directly
    if isinstance(value, pd.Timestamp):
        ts = value
    else:
        s = str(value).strip()

        # Slack often uses epoch seconds like "1700000000.1234"
        if re.fullmatch(r"\d{10}(\.\d+)?", s):
            sec = float(s)
            return pd.to_datetime(sec, unit="s", utc=True)
        # Try common date formats (ISO, calendar exports, etc.)
        ts = pd.to_datetime(s, utc=True, errors="coerce")
  # If parsing failed, pandas returns NaT (“Not a Time”)
    if ts is pd.NaT:
        return None
# Ensure UTC (timezone-aware)
    if ts.tzinfo is None:
        ts = ts.tz_localize(timezone.utc)
    else:
        ts = ts.tz_convert(timezone.utc)

    return ts


def ensure_dir(p: Path) -> None:
    """
    Makes sure an output folder exists.
    If it doesn’t exist, this creates it.
    """
    p.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2) OTQ-style scoring
# ============================================================

def otq_event_units(event_type: str, duration_min: float, participant_count: int) -> float:
    """
    Turn one communication event into an engagement score (EU).

    Customize this function to match your OTQ policy.
    """
    t = (event_type or "").lower()

    # Base engagement points by interaction type (defaults)
    if "meeting" in t or "teams_meeting" in t:
        base = 24.5
    elif "call" in t or "phone" in t:
        base = 8.5
    elif "email" in t or "outlook" in t:
        base = 6.0
    elif "chat" in t or "message" in t or "slack" in t:
        base = 9.5
    else:
        base = 5.0

    # Duration adds a small bonus (capped at 120 minutes => max bonus of 2)
    dur_bonus = min(max(duration_min, 0.0), 120.0) / 60.0

    # Participants add bonus with diminishing returns (log scale)
    pc = max(participant_count, 1)
    participant_bonus = math.log(pc, 2) * 1.5

    return float(base + dur_bonus + participant_bonus)


# ============================================================
# 3) Loading functions (CSV / JSON / JSONL)
# ============================================================

def load_csv_any(path: Path) -> pd.DataFrame:
    """Load CSV as strings, normalize later."""
    return pd.read_csv(path, dtype=str, keep_default_na=False)


def load_json_any(path: Path) -> Any:
    """
    Load JSON or JSONL.
    - JSON: a dict or list
    - JSONL: one JSON object per line
    """
    text = path.read_text(encoding="utf-8", errors="ignore").strip()
    if not text:
        return []

    if text.lstrip().startswith("{") or text.lstrip().startswith("["):
        return json.loads(text)

    rows = []
    for line in text.splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
    return rows


# ============================================================
# 4) Parsers (Slack / Outlook / Teams) -> standard event table
#    Standard columns:
#      source, event_type, timestamp, actor, channel, text, duration_min, participant_count
# ============================================================

def parse_slack_file(path: Path) -> pd.DataFrame:
    """
    Slack parser:
    - CSV: tries to locate ts/text/user/channel columns
    - JSON/JSONL: reads Slack-like message objects
    """
    if path.suffix.lower() == ".csv":
        df = load_csv_any(path)

        ts_col = next((c for c in df.columns if c.lower() in ["ts", "timestamp", "time", "date"]), None)
        text_col = next((c for c in df.columns if c.lower() in ["text", "message", "body"]), None)
        user_col = next((c for c in df.columns if c.lower() in ["user", "username", "from"]), None)
        chan_col = next((c for c in df.columns if c.lower() in ["channel", "room", "thread"]), None)

        out = pd.DataFrame({
            "source": "slack",
            "event_type": "chat",
            "timestamp": df[ts_col] if ts_col else None,
            "actor": df[user_col] if user_col else None,
            "channel": df[chan_col] if chan_col else None,
            "text": df[text_col] if text_col else None,
            "duration_min": 0.0,
            "participant_count": 2,
        })
        return out

    data = load_json_any(path)
    rows = []

    if isinstance(data, dict):
        data = [data]

    for obj in data:
        if not isinstance(obj, dict):
            continue

        ts = obj.get("ts") or obj.get("timestamp") or obj.get("time")
        user = obj.get("user") or obj.get("username")
        text = obj.get("text") or obj.get("message")
        channel = obj.get("channel") or obj.get("room")

        rows.append({
            "source": "slack",
            "event_type": "chat",
            "timestamp": ts,
            "actor": user,
            "channel": channel,
            "text": text,
            "duration_min": 0.0,
            "participant_count": 2,
        })

    return pd.DataFrame(rows)


def parse_outlook_calendar_file(path: Path) -> pd.DataFrame:
    """
    Outlook Calendar CSV parser.

    Creates meeting events with a participant count estimate.
    """
    df = load_csv_any(path)
    cols = {c.lower(): c for c in df.columns}

    subj = df[cols["subject"]] if "subject" in cols else ""
    start = df[cols["start"]] if "start" in cols else (df[cols["start time"]] if "start time" in cols else "")
    end = df[cols["end"]] if "end" in cols else (df[cols["end time"]] if "end time" in cols else "")
    org = df[cols["organizer"]] if "organizer" in cols else ""
    req = df[cols["required attendees"]] if "required attendees" in cols else ""
    opt = df[cols["optional attendees"]] if "optional attendees" in cols else ""

    def count_people(s_req: str, s_opt: str) -> int:
        def split_people(s: str) -> List[str]:
            s = (s or "").strip()
            if not s:
                return []
            parts = re.split(r"[;,]", s)
            return [p.strip() for p in parts if p.strip()]

        ppl = split_people(s_req) + split_people(s_opt)
        return max(1, len(set(ppl)) + 1)  # +1 for organizer

    part_counts = [count_people(r, o) for r, o in zip(req.tolist(), opt.tolist())]

    out = pd.DataFrame({
        "source": "outlook",
        "event_type": "meeting",
        "timestamp": start,
        "actor": org,
        "channel": "calendar",
        "text": subj,
        "start": start,
        "end": end,
        "duration_min": np.nan,  # duration could be computed from start/end if desired
        "participant_count": part_counts,
    })
    return out


def parse_teams_file(path: Path) -> pd.DataFrame:
    """
    Teams CSV parser (flexible / best-effort).
    """
    df = load_csv_any(path)
    cols = {c.lower(): c for c in df.columns}

    ts_col = next((cols[k] for k in cols if k in ["timestamp", "time", "date", "start", "start time", "start_time"]), None)
    subject_col = next((cols[k] for k in cols if k in ["subject", "title", "meeting subject", "name"]), None)
    organizer_col = next((cols[k] for k in cols if k in ["organizer", "from", "owner"]), None)
    duration_col = next((cols[k] for k in cols if k in ["duration", "duration_min", "minutes"]), None)
    participants_col = next((cols[k] for k in cols if k in ["participants", "attendees", "participant_count"]), None)
    type_col = next((cols[k] for k in cols if k in ["type", "interactiontype", "event_type"]), None)

    event_type = df[type_col] if type_col else "teams_meeting"

    out = pd.DataFrame({
        "source": "teams",
        "event_type": event_type,
        "timestamp": df[ts_col] if ts_col else None,
        "actor": df[organizer_col] if organizer_col else None,
        "channel": "teams",
        "text": df[subject_col] if subject_col else None,
        "duration_min": df[duration_col] if duration_col else 0.0,
        "participant_count": df[participants_col] if participants_col else 2,
    })
    return out


def discover_inputs(input_dir: Path) -> List[Path]:
    """Find all csv/json/jsonl files under the input folder."""
    exts = {".csv", ".json", ".jsonl"}
    return [p for p in input_dir.rglob("*") if p.is_file() and p.suffix.lower() in exts]


def parse_any(path: Path, debug: bool=False) -> Optional[pd.DataFrame]:
    """
    Route files to the right parser based on filename keywords.
    """
    name = path.name.lower()

    try:
        if "slack" in name:
            if debug: print(f"[INFO] Parsing Slack file: {path}")
            return parse_slack_file(path)

        if "outlook" in name or "calendar" in name:
            if path.suffix.lower() == ".csv":
                if debug: print(f"[INFO] Parsing Outlook Calendar file: {path}")
                return parse_outlook_calendar_file(path)
            return None

        if "teams" in name:
            if path.suffix.lower() == ".csv":
                if debug: print(f"[INFO] Parsing Teams file: {path}")
                return parse_teams_file(path)
            return None

        # Unknown CSV: try Teams-style best effort
        if path.suffix.lower() == ".csv":
            if debug: print(f"[INFO] Parsing Unknown CSV as Teams-like: {path}")
            return parse_teams_file(path)

        return None

    except Exception as e:
        print(f"[WARN] Failed parsing {path}: {e}")
        return None


# ============================================================
# 5) Feature engineering
# ============================================================

def build_event_table(dfs: List[pd.DataFrame]) -> pd.DataFrame:
    """
    Stitch parsed tables into one master event table, then:
    - normalize timestamps
    - compute daily buckets
    - compute EU score
    - add text length
    """
    df = pd.concat(dfs, ignore_index=True)

    df["ts"] = df["timestamp"].apply(parse_datetime)
    df = df.dropna(subset=["ts"]).copy()

    df["date"] = df["ts"].dt.floor("D")

    df["duration_min"] = df["duration_min"].apply(safe_float)
    df["participant_count"] = df["participant_count"].apply(safe_int)

    df["eu"] = [
        otq_event_units(t, d, p)
        for t, d, p in zip(df["event_type"].astype(str), df["duration_min"], df["participant_count"])
    ]

    df["text_len"] = df["text"].fillna("").astype(str).str.len()

    return df


def make_daily_features(events: pd.DataFrame) -> pd.DataFrame:
    """One row per day, summarizing engagement and activity."""
    g = events.groupby("date", as_index=False).agg(
        total_events=("eu", "size"),
        total_eu=("eu", "sum"),
        avg_eu=("eu", "mean"),
        avg_participants=("participant_count", "mean"),
        total_text=("text_len", "sum"),
        avg_duration=("duration_min", "mean"),
    )

    g["log_total_eu"] = np.log1p(g["total_eu"])
    g["log_total_events"] = np.log1p(g["total_events"])
    g["log_total_text"] = np.log1p(g["total_text"].clip(lower=0))

    return g


def compute_outcome_proxy(events: pd.DataFrame) -> pd.DataFrame:
    """
    Proxy outcome: median minutes between consecutive events in same channel stream.
    """
    e = events.sort_values("ts").copy()
    e["channel_key"] = e["source"].astype(str) + "::" + e["channel"].astype(str)

    e["prev_ts"] = e.groupby("channel_key")["ts"].shift(1)
    e["delta_min"] = (e["ts"] - e["prev_ts"]).dt.total_seconds() / 60.0

    # Filter to reasonable response gaps (0 < gap <= 12 hours)
    e = e[(e["delta_min"].notna()) & (e["delta_min"] > 0) & (e["delta_min"] <= 720)].copy()

    rt = e.groupby("date", as_index=False).agg(
        median_response_min=("delta_min", "median"),
        p90_response_min=("delta_min", lambda s: np.percentile(s, 90)),
        samples=("delta_min", "size"),
    )
    return rt


# ============================================================
# 6) Modeling + plotting
# ============================================================

def run_regression(daily: pd.DataFrame, outcome: pd.DataFrame, out_dir: Path) -> pd.DataFrame:
    """
    Merge daily features + outcome, run Ridge regression,
    save plots and return merged table with predictions.
    """
    df = daily.merge(outcome, on="date", how="inner").copy()

    if df.empty or len(df) < 10:
        print("[ERROR] Not enough merged daily rows to run regression (need ~10+).")
        return df

    feature_cols = [
        "log_total_eu",
        "log_total_events",
        "avg_participants",
        "avg_duration",
        "log_total_text",
    ]
    target_col = "median_response_min"

    X = df[feature_cols].fillna(0.0).to_numpy()
    y = df[target_col].fillna(df[target_col].median()).to_numpy()

    X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
        X, y, df, test_size=0.25, random_state=42
    )

    model = Ridge(alpha=1.0)
    model.fit(X_train, y_train)

    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    print("\n=== Regression Results ===")
    print(f"Train R^2: {r2_score(y_train, pred_train):.3f}")
    print(f"Test  R^2: {r2_score(y_test, pred_test):.3f}")
    print(f"Test  MAE: {mean_absolute_error(y_test, pred_test):.2f} minutes")

    coefs = pd.DataFrame({"feature": feature_cols, "coef": model.coef_}).sort_values("coef", ascending=False)
    print("\nCoefficients:")
    display(coefs)

    ensure_dir(out_dir)

    # Plot 1: Actual vs Predicted (test)
    plt.figure()
    plt.scatter(y_test, pred_test)
    plt.xlabel("Actual median response (min)")
    plt.ylabel("Predicted median response (min)")
    plt.title("Actual vs Predicted (Test Set)")
    p1 = out_dir / "actual_vs_predicted.png"
    plt.savefig(p1, dpi=150, bbox_inches="tight")
    plt.close()

    # Plot 2: Trend over time
    df_all = df.sort_values("date").copy()
    df_all["pred"] = model.predict(df_all[feature_cols].fillna(0.0).to_numpy())

    plt.figure()
    plt.plot(df_all["date"], df_all[target_col], label="Actual")
    plt.plot(df_all["date"], df_all["pred"], label="Predicted")
    plt.xlabel("Date")
    plt.ylabel("Median response (min)")
    plt.title("Outcome Trend: Actual vs Predicted")
    plt.legend()
    p2 = out_dir / "trend_actual_vs_predicted.png"
    plt.savefig(p2, dpi=150, bbox_inches="tight")
    plt.close()

    # Plot 3: EU vs Outcome
    plt.figure()
    plt.scatter(df_all["total_eu"], df_all[target_col])
    plt.xlabel("Total engagement score (EU)")
    plt.ylabel("Median response (min)")
    plt.title("Engagement (EU) vs Outcome (Response Time)")
    p3 = out_dir / "eu_vs_outcome.png"
    plt.savefig(p3, dpi=150, bbox_inches="tight")
    plt.close()

    df_all.to_csv(out_dir / "daily_model_table.csv", index=False)

    print("\nSaved outputs:")
    print(f" - {p1}")
    print(f" - {p2}")
    print(f" - {p3}")
    print(f" - {out_dir / 'daily_model_table.csv'}")

    return df_all


def run_pipeline(input_dir: str, out_dir: str, debug: bool=False):
    """
    One-call runner for the whole notebook pipeline.
    """
    input_path = Path(input_dir).expanduser().resolve()
    out_path = Path(out_dir).expanduser().resolve()

    if not input_path.exists():
        raise FileNotFoundError(f"Input folder not found: {input_path}")

    paths = discover_inputs(input_path)
    if not paths:
        raise FileNotFoundError(f"No CSV/JSON found under: {input_path}")

    parsed = []
    for p in paths:
        df = parse_any(p, debug=debug)
        if df is not None and not df.empty:
            parsed.append(df)

    if not parsed:
        raise RuntimeError("No parsable files found. Rename files to include 'teams', 'outlook', or 'slack'.")

    events = build_event_table(parsed)
    daily = make_daily_features(events)
    outcome = compute_outcome_proxy(events)

    merged = run_regression(daily, outcome, out_path)

    return events, daily, outcome, merged


In [ ]:
# =========================
# Run the pipeline
# =========================

events, daily, outcome, merged = run_pipeline(INPUT_DIR, OUT_DIR, debug=DEBUG)

print("\nEvents table preview:")
display(events.head())

print("\nDaily features preview:")
display(daily.head())

print("\nOutcome preview:")
display(outcome.head())

print("\nMerged modeling table preview:")
display(merged.head())


FileNotFoundError: Input folder not found: /content/data